# Avito Services: Candidate Generation

# Архитектура

Принципиальные отличия от v1-v3:

## 1. Несколько независимых источников кандидатов -> объединение (не один скор)
    bm25          top-200   — лексический поиск без гео
    bm25_geo      top-200   — BM25 * (1 + 1.25 * affinity), близкие выше
    char          top-100   — символьный TF-IDF (3-5-граммы), устойчив к опечаткам
    char_geo      top-100   — char * (1 + affinity)
    dense         top-200   — косинусное сходство (e5-small/base fine-tuned)
    dense_geo     top-200   — dense + гео-бонус
    filter_bm25   top-100   — запрос из фильтров vs параметры объявлений
    lookup        все       — история кликов
    ИТОГО union: ~600 кандидатов, полнота пула ~0.97

## 2. Гео — маленький признак, НЕ главный фильтр
    affinity = max(same_location, loc_transition_prob, 0.8*exp(-dist/50km))
    geo_boost  = 0.08 * affinity                            # <= +0.08
    geo_penalty = -0.012 * min(log(1 + dist/30), 6)        # >= -0.072
    geo_reward  = geo_boost + geo_penalty
    Для BM25: bm25_geo = bm25 * (1 + 1.25 * affinity)  — мультипликативно
    Для dense: dense_geo = dense + geo_reward            — аддитивно
    В предыдущих версиях вес гео был 0.50 — доминировал над текстом и dense.

## 3. CatBoostRanker PairLogit (не Classifier!)
    Группировка по запросу, оптимизация порядка внутри группы.

## 4. Два прохода обучения (с hard negatives)
    Первый: top-128 по RRF + 128 случайных как негативы.
    Второй: top-128 по прогнозу первой модели как hard negatives.


## 1. Установка зависимостей

In [2]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu snowballstemmer pyarrow tqdm catboost scikit-learn scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


## 2. Загрузка данных

In [3]:
import os, requests
from pathlib import Path

PUBLIC_URL = "https://disk.yandex.ru/d/sNhfo0YOjGtufg"
r = requests.get(f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={PUBLIC_URL}")
if r.status_code != 200: raise RuntimeError(str(r.status_code))
os.system(f'wget -q --show-progress -O avito.zip "{r.json()["href"]}"')
p0 = Path("/content/first"); p0.mkdir(exist_ok=True)
os.system(f"unzip -q -o avito.zip -d {p0}")
DATA_DIR = Path("/content/data"); DATA_DIR.mkdir(exist_ok=True)
os.system(f"unzip -q -o /content/first/NLP_avito_interns/dataset.zip -d {DATA_DIR}")
for p in list(DATA_DIR.rglob("*.parquet")):
    t = DATA_DIR / p.name
    if not t.exists(): p.rename(t)
print("Данные:"); os.system(f"ls -lh {DATA_DIR}/*.parquet")

Данные:


0

## 3. Импорты и параметры

In [4]:
import re, gc, html, time, warnings
from math import radians, sin, cos, sqrt, atan2
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from tqdm.auto import tqdm
import snowballstemmer
warnings.filterwarnings("ignore")

try:
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
    from torch.utils.data import DataLoader
    import faiss
    HAS_DENSE = True; print("[OK] dense retrieval")
except ImportError:
    HAS_DENSE = False; print("[INFO] pip install sentence-transformers faiss-cpu")

try:
    from catboost import CatBoostRanker, Pool as CatPool
    HAS_CB = True; print("[OK] catboost")
except ImportError:
    HAS_CB = False; print("[INFO] pip install catboost")

SEED   = 42; DATA_DIR = Path("/content/data"); SAVE_DIR = Path("/content/artifacts")
SAVE_DIR.mkdir(exist_ok=True)
N_OUT  = 50; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Параметры ретривера
TOP = {"bm25": 200, "bm25_geo": 200, "char": 100, "char_geo": 100,
       "dense": 200, "dense_geo": 200, "filter_bm25": 100}

# Параметры BM25
BM25_K1, BM25_B = 1.2, 0.65
WORD_VOCAB  = 300_000
CHAR_VOCAB  = 180_000
DESC_CHARS  = 2400
PARAMS_CHARS = 900

# Параметры ранкера
RANKER = {"depth": 6, "learning_rate": 0.06, "iterations": 300}
HARD_NEG, RAND_NEG = 128, 128

# Dense
DENSE_MODEL  = "intfloat/multilingual-e5-small"
DENSE_MAXLEN = 224
DENSE_BATCH  = 64
ENC_BATCH    = 256

np.random.seed(SEED); torch.manual_seed(SEED)

[OK] dense retrieval
[OK] catboost
Device: cuda


## 4. Загрузка данных

In [5]:
print("Загружаем данные...")
train             = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_items   = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

for df, col in [(benchmark_items,"item_id"),(benchmark_queries,"query_id"),(train,"item_id")]:
    df[col] = df[col].astype(str)

# Числовые поля
for df in [train, benchmark_items]:
    for col in ["item_price","item_rating","item_rating_reviews_count","item_latitude","item_longitude"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
    for col in ["item_title_raw","item_description_raw","item_infm_params_text"]:
        if col in df.columns:
            df[col] = df[col].fillna("")

for df in [train, benchmark_queries]:
    if "search_infm_params_text" not in df.columns:
        df["search_infm_params_text"] = ""
    else:
        df["search_infm_params_text"] = df["search_infm_params_text"].fillna("")

benchmark_items = benchmark_items.sort_values("item_id").reset_index(drop=True)
VALID_IDS = set(benchmark_items["item_id"])
ALL_IDS   = benchmark_items["item_id"].tolist()
N_ITEMS   = len(benchmark_items)
print(f"train: {len(train):,} | items: {N_ITEMS:,} | queries: {len(benchmark_queries):,}")

Загружаем данные...
train: 497,673 | items: 189,212 | queries: 2,452


## 5. Нормализация текста и стемминг

In [6]:
_VISUAL = str.maketrans("aceopxykmt", "асеорхукмт")
_MIX    = re.compile(r"\b(?=[a-zа-я]*[а-я])(?=[a-zа-я]*[a-z])[a-zа-я]+\b")
_stemmer = snowballstemmer.stemmer("russian")

def normalize(text, fix_visual=False):
    """Нижний регистр, е->е, HTML-сущности, опционально замена визуально похожих букв."""
    text = re.sub(r"<[^>]+>", " ", html.unescape(str(text))).lower().replace("е", "е")
    if fix_visual:
        text = _MIX.sub(lambda m: m.group().translate(_VISUAL), text)
    return " ".join(re.findall(r"[a-zа-я0-9]+", text))

def stem_tokenize(text):
    """Стемминг + токенизация для CountVectorizer."""
    return [_stemmer.stemWord(w) for w in normalize(text).split()]

# Нормализуем тексты запросов
for df in [train, benchmark_queries]:
    df["qnorm"] = df["search_query"].map(lambda t: normalize(t))

## 6. Разделение: core (история) / held-out (валидация)

In [7]:
# held-out запросы исключены из истории — CatBoost не видит их при обучении
rng = np.random.default_rng(SEED)
held_texts = rng.choice(train["qnorm"].unique(), size=min(3000, train["qnorm"].nunique()),
                         replace=False)
held_set  = set(held_texts)
core      = train[~train["qnorm"].isin(held_set)].copy()
held_rows = train[train["qnorm"].isin(held_set)]

# Validation: 500 запросов из held (знакомые тексты, но исключены из истории)
val_texts = rng.choice(held_texts, size=min(500, len(held_texts)), replace=False)
val_set   = set(val_texts)
val_rows  = held_rows[held_rows["qnorm"].isin(val_set)]

val_ood = (val_rows.groupby("search_query")["item_id"]
           .apply(lambda x: sorted(set(x) & VALID_IDS))
           .reset_index())
val_ood.columns = ["search_query", "relevant_ids"]
val_ood = val_ood[val_ood["relevant_ids"].apply(len) > 0]
print(f"Core: {len(core):,} | Held: {len(held_rows):,} | Val: {len(val_ood):,} запросов")

Core: 479,710 | Held: 17,963 | Val: 67 запросов


## 7. Лексический индекс: BM25 по полям + Char TF-IDF

In [8]:
def make_bm25_matrix(count_mat, k1=BM25_K1, b=BM25_B):
    """
    Преобразуем матрицу частот (n_items × vocab) в BM25-матрицу (vocab × n_items).
    query_vec @ bm25_matrix = BM25-скоры для всех n_items.
    """
    cm = count_mat.tocsr().astype(np.float32)
    lengths = np.asarray(cm.sum(axis=1)).ravel()
    freq    = np.bincount(cm.indices, minlength=cm.shape[1]).astype(np.float32)
    idf     = np.log1p((cm.shape[0] - freq + 0.5) / (freq + 0.5))
    avgdl   = max(float(lengths.mean()), 1.0)
    norm    = k1 * (1 - b + b * lengths / avgdl)
    rows    = np.repeat(np.arange(cm.shape[0]), np.diff(cm.indptr))
    cm.data = cm.data * (k1 + 1) / (cm.data + norm[rows]) * idf[cm.indices]
    return cm.T.tocsr()   # vocab × n_items

print("Строим лексический индекс...")
t0 = time.time()

# Тексты полей корпуса
title_texts  = benchmark_items["item_title_raw"].map(normalize).tolist()
desc_texts   = benchmark_items["item_description_raw"].str[:DESC_CHARS].map(normalize).tolist()
params_texts = benchmark_items["item_infm_params_text"].str[:PARAMS_CHARS].map(normalize).tolist()

# Общий словарь по всем полям
vocab_vec = CountVectorizer(
    tokenizer=stem_tokenize, token_pattern=None, lowercase=False,
    ngram_range=(1, 2), min_df=2, max_features=WORD_VOCAB, dtype=np.float32,
)
combined_texts = [f"{t} {d} {p}" for t,d,p in zip(title_texts, desc_texts, params_texts)]
vocab_vec.fit(tqdm(combined_texts, desc="  словарь", leave=False))

# BM25 по каждому полю
print("  BM25 по полям (title, desc, params)...")
title_bm25  = make_bm25_matrix(vocab_vec.transform(title_texts))
desc_bm25   = make_bm25_matrix(vocab_vec.transform(desc_texts))
params_bm25 = make_bm25_matrix(vocab_vec.transform(params_texts))

# Char TF-IDF по заголовкам (устойчив к опечаткам и морфологии)
print("  Char TF-IDF...")
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                            max_features=CHAR_VOCAB, sublinear_tf=True, dtype=np.float32)
char_mat = char_vec.fit_transform(title_texts).T.tocsr()   # char_vocab × n_items
print(f"  Время: {time.time()-t0:.1f}с | BM25 vocab: {title_bm25.shape[0]:,} | char vocab: {char_mat.shape[0]:,}")

def bm25_query_scores(query_text):
    """BM25-скор для одного запроса по всем объявлениям: 3*title + 1*desc + 0.6*params."""
    qvec = vocab_vec.transform([normalize(query_text)])
    t = np.asarray((qvec @ title_bm25).todense()).ravel()
    d = np.asarray((qvec @ desc_bm25).todense()).ravel()
    p = np.asarray((qvec @ params_bm25).todense()).ravel()
    return 3 * t + d + 0.6 * p

def char_query_scores(query_text):
    """Char TF-IDF скор."""
    qchar = char_vec.transform([normalize(query_text)])
    return np.asarray((qchar @ char_mat).todense()).ravel()

Строим лексический индекс...


  словарь:   0%|          | 0/189212 [00:00<?, ?it/s]

  BM25 по полям (title, desc, params)...
  Char TF-IDF...
  Время: 2996.0с | BM25 vocab: 300,000 | char vocab: 97,268


## 8. Гео: центроиды, переходные вероятности, affinity

Принципиальное отличие от v1-v3:
Гео НЕ является основным фильтром пула.
affinity = max(same_location, transition_prob, 0.8*exp(-dist/50km))
geo_reward = 0.08*affinity - 0.012*min(log(1+dist/30), 6)
Максимальный положительный вклад: +0.08 (не 0.50!)
Максимальный штраф: -0.072 (для очень далеких объявлений)
BM25_geo = BM25 * (1 + 1.25 * affinity) — мультипликативно

In [9]:
print("Строим гео-индекс...")
_lats = pd.to_numeric(benchmark_items["item_latitude"],  errors="coerce").values.astype(np.float32)
_lons = pd.to_numeric(benchmark_items["item_longitude"], errors="coerce").values.astype(np.float32)
_ok   = ~np.isnan(_lats) & ~np.isnan(_lons) & (_lats != 0) & (_lons != 0)

# Координаты объявлений в радианах
item_lat_rad = np.where(_ok, np.deg2rad(_lats), np.nan).astype(np.float32)
item_lon_rad = np.where(_ok, np.deg2rad(_lons), np.nan).astype(np.float32)

item_loc_id  = benchmark_items["item_location_id"].fillna(-1).astype(int).values

# Центры локаций: медиана координат объявлений с данным item_location_id
loc_centers = {}    # loc_id -> (lat_deg, lon_deg)
if "item_location_id" in benchmark_items.columns:
    tmp = pd.DataFrame({"loc": item_loc_id, "lat": _lats, "lon": _lons})
    tmp = tmp[(tmp.lat != 0) & (~tmp.lat.isna())]
    for loc, g in tmp.groupby("loc"):
        loc_centers[int(loc)] = (float(g.lat.median()), float(g.lon.median()))
print(f"  Центров локаций: {len(loc_centers):,}")

# Переходные вероятности: P(item_location | search_location) из core
loc_trans = {}   # search_loc -> {item_loc: prob}
if "search_location_id" in core.columns and "item_location_id" in core.columns:
    trans = core.dropna(subset=["search_location_id","item_location_id"])
    counts = trans.groupby(["search_location_id","item_location_id"]).size()
    totals = counts.groupby(level=0).sum()
    probs  = counts / totals
    for (sloc, iloc), prob in probs.items():
        loc_trans.setdefault(int(sloc), {})[int(iloc)] = float(prob)
print(f"  Переходных вероятностей: {sum(len(v) for v in loc_trans.values()):,}")

# Словарь item_loc_id -> list of row indices
_loc_to_rows = defaultdict(list)
for i, loc in enumerate(item_loc_id):
    _loc_to_rows[int(loc)].append(i)
loc_to_rows = dict(_loc_to_rows)

def compute_geo(search_loc_id):
    """
    Вычислить geo-признаки для ВСЕХ объявлений заданной локации поиска.
    Возвращает (same, probability, distance_km, affinity, geo_reward) — vectors n_items.
    """
    sloc = int(search_loc_id) if search_loc_id and not (isinstance(search_loc_id, float) and np.isnan(search_loc_id)) else -1

    same        = (item_loc_id == sloc).astype(np.float32)
    probability = np.zeros(N_ITEMS, dtype=np.float32)
    distance    = np.full(N_ITEMS, np.nan, dtype=np.float32)

    # Заполняем переходные вероятности
    if sloc in loc_trans:
        for iloc, prob in loc_trans[sloc].items():
            rows = loc_to_rows.get(iloc, [])
            if rows: probability[rows] = prob

    # Вычисляем расстояния (Haversine на все объявления сразу)
    if sloc in loc_centers:
        clat_r, clon_r = np.deg2rad(loc_centers[sloc][0]), np.deg2rad(loc_centers[sloc][1])
        h = (np.sin((item_lat_rad - clat_r) / 2) ** 2
             + np.cos(item_lat_rad) * np.cos(clat_r)
             * np.sin((item_lon_rad - clon_r) / 2) ** 2)
        distance = np.where(~np.isnan(item_lat_rad),
                            12742 * np.arcsin(np.sqrt(np.clip(h, 0, 1))), np.nan)

    # affinity: максимум из трех сигналов
    max_prob  = float(probability.max()) if probability.max() > 0 else 1e-8
    dist_aff  = 0.8 * np.exp(-np.nan_to_num(distance, nan=1e6) / 50)
    affinity  = np.maximum.reduce([same, probability / max_prob, dist_aff])

    # geo_reward: маленький бонус/штраф (максимум +0.08, минимум -0.072)
    geo_reward = (0.08 * affinity
                  - 0.012 * np.minimum(np.log1p(np.nan_to_num(distance, nan=0) / 30), 6))

    return same, probability, distance, affinity, geo_reward

def _get_loc(q_row):
    v = q_row.get("search_location_id", None)
    if v is None or (isinstance(v, float) and np.isnan(v)): return -1
    return int(v)

Строим гео-индекс...
  Центров локаций: 2,877
  Переходных вероятностей: 16,043


## 9. Train Lookup (история кликов)

In [10]:
print("Строим lookup...")
prod_gl  = defaultdict(Counter)   # query_norm -> Counter(item_id)
prod_cat = defaultdict(Counter)   # (qnorm, cat) -> Counter(item_id)
prod_loc = defaultdict(Counter)   # (qnorm, loc) -> Counter(item_id)
for _, r in tqdm(core.iterrows(), total=len(core), desc="  lookup", leave=False):
    if r["item_id"] not in VALID_IDS: continue
    q = r["qnorm"]; iid = r["item_id"]
    prod_gl[q][iid]  += 1
    cat = str(r.get("search_category","") or "")
    prod_cat[(q, cat)][iid] += 1
    loc = str(int(r.get("search_location_id",-1) or -1))
    if loc != "-1": prod_loc[(q, loc)][iid] += 1

item_pop = Counter()
for ctr in prod_gl.values(): item_pop.update(ctr)
item_known = set(item_pop.keys())
print(f"  Уникальных q в lookup: {len(prod_gl):,} | popular items: {len(item_pop):,}")

Строим lookup...


  lookup:   0%|          | 0/479710 [00:00<?, ?it/s]

  Уникальных q в lookup: 11,713 | popular items: 17,704


## 10. Dense Retrieval

In [11]:
dense_model = None; item_emb = None; faiss_idx = None

if HAS_DENSE:
    print(f"\nЗагружаем {DENSE_MODEL}...")
    dense_model = SentenceTransformer(DENSE_MODEL, device=DEVICE)
    dense_model.max_seq_length = DENSE_MAXLEN
    emb_dim = dense_model.get_sentence_embedding_dimension()
    print(f"  dim={emb_dim}")

    if torch.cuda.is_available():
        print("Fine-tuning (CachedMNRL, 1 проход)...")
        # Обучение: пары (запрос, объявление) из core без overlap с val
        train_pairs = core[core["item_id"].isin(VALID_IDS)].drop_duplicates(["qnorm","item_id"])
        # 1 уникальный запрос = 1 пара (контрастивное обучение: каждый текст 1 раз)
        train_pairs = (train_pairs.sample(frac=1, random_state=SEED)
                       .drop_duplicates("qnorm")
                       .head(60_000))
        q_texts = ("query: " + train_pairs["search_query"].fillna("")).tolist()
        i_texts = ("passage: " + train_pairs["item_title_raw"].fillna("") + " "
                   + train_pairs["item_infm_params_text"].fillna("").str[:350] + " "
                   + train_pairs["item_description_raw"].fillna("").str[:1800]).tolist()
        print(f"  Fine-tune пар: {len(q_texts):,}")

        # Обучение с CachedMultipleNegativesRankingLoss
        examples = [InputExample(texts=[q, p]) for q, p in zip(q_texts, i_texts)]
        loader = DataLoader(examples, shuffle=True, batch_size=DENSE_BATCH, drop_last=True)
        try:
            loss_fn = CachedMultipleNegativesRankingLoss(dense_model, mini_batch_size=16)
        except Exception:
            loss_fn = losses.MultipleNegativesRankingLoss(dense_model)
        dense_model.fit(
            train_objectives=[(loader, loss_fn)],
            epochs=1, warmup_steps=int(len(loader)*0.1),
            show_progress_bar=True,
            output_path=str(SAVE_DIR/"dense_ft"),
        )
        dense_model = SentenceTransformer(str(SAVE_DIR/"dense_ft"), device=DEVICE)
        dense_model.max_seq_length = DENSE_MAXLEN
        print("  Fine-tuning завершено")
    else:
        print("  GPU нет — pre-trained")

    print(f"Кодируем {N_ITEMS:,} объявлений...")
    t0 = time.time()
    corpus_texts = [
        f"passage: {r.item_title_raw} {r.item_infm_params_text[:350]} {r.item_description_raw[:1800]}"
        for _, r in benchmark_items.iterrows()
    ]
    item_emb = dense_model.encode(
        corpus_texts, batch_size=ENC_BATCH, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True, device=DEVICE,
    ).astype("float32")
    faiss_idx = faiss.IndexFlatIP(emb_dim); faiss_idx.add(item_emb)
    np.save(SAVE_DIR/"item_emb.npy", item_emb)
    faiss.write_index(faiss_idx, str(SAVE_DIR/"faiss.bin"))
    print(f"  {item_emb.shape}, {time.time()-t0:.1f}с")
else:
    print("[WARN] Dense недоступен")


Загружаем intfloat/multilingual-e5-small...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Fine-tuning завершено
Кодируем 189,212 объявлений...


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

  (189212, 384), 1120.6с


## 11. Вспомогательные функции: пул кандидатов + признаки

In [12]:
def topk_indices(scores, k):
    """Индексы top-k по убыванию скора."""
    k = min(k, len(scores))
    if k == 0: return np.empty(0, dtype=np.int32)
    part = np.argpartition(scores, -k)[-k:]
    return part[np.argsort(-scores[part])].astype(np.int32)


def rrf_score(rank_dicts, k=60):
    """Reciprocal Rank Fusion: sum(1/(k+rank)) по всем источникам."""
    scores = defaultdict(float)
    for rdict in rank_dicts:
        for idx, rank in rdict.items():
            scores[idx] += 1.0 / (k + rank)
    return scores


def query_dense_scores(q_emb):
    """Косинусные оценки для всех объявлений."""
    if item_emb is None: return np.zeros(N_ITEMS)
    return (item_emb @ q_emb).ravel()


def build_candidate_pool(q_row, return_scores=False):
    """
    Собрать пул кандидатов из нескольких источников.

    Возвращает: (pool_item_ids, score_dict)
    score_dict: {source: array[N_ITEMS]} — скоры для признаков CatBoost
    """
    q_text = str(q_row.get("search_query",""))
    q_norm = normalize(q_text)
    q_cat  = str(q_row.get("search_category","") or "")
    q_loc  = _get_loc(q_row)
    q_infm = str(q_row.get("search_infm_params_text","") or "")

    # Гео (векторы по всем объявлениям сразу)
    same, prob, dist, aff, geo_rew = compute_geo(q_loc)

    # --- Лексика ---
    bm25_s      = bm25_query_scores(q_text)
    bm25_geo_s  = bm25_s * (1 + 1.25 * aff)
    char_s      = char_query_scores(q_text)
    char_geo_s  = char_s * (1 + aff)
    filter_s    = bm25_query_scores(q_infm) if q_infm.strip() else np.zeros(N_ITEMS)

    # --- Dense ---
    dense_s     = np.zeros(N_ITEMS, dtype=np.float32)
    dense_geo_s = np.zeros(N_ITEMS, dtype=np.float32)
    q_emb_vec   = None
    if dense_model is not None and item_emb is not None:
        q_emb_vec   = dense_model.encode([f"query: {q_text}"], normalize_embeddings=True,
                                          show_progress_bar=False, convert_to_numpy=True,
                                          device=DEVICE)[0].astype("float32")
        dense_s     = query_dense_scores(q_emb_vec)
        dense_geo_s = dense_s + geo_rew

    # --- Кандидаты из каждого источника ---
    source_scores = {
        "bm25":       bm25_s,
        "bm25_geo":   bm25_geo_s,
        "char":       char_s,
        "char_geo":   char_geo_s,
        "dense":      dense_s,
        "dense_geo":  dense_geo_s,
        "filter_bm25": filter_s,
    }

    rank_dicts = []
    pool_set   = set()
    for src, scores_arr in source_scores.items():
        k = TOP.get(src, 100)
        if scores_arr.max() <= 0: continue
        idxs = topk_indices(scores_arr, k)
        rank_dicts.append({int(idx): rank+1 for rank, idx in enumerate(idxs)})
        pool_set.update(int(i) for i in idxs if ALL_IDS[int(i)] in VALID_IDS)

    # --- Lookup ---
    lu_ids = []
    for key, lu_dict in [(q_norm, prod_gl), ((q_norm, q_cat), prod_cat),
                          ((q_norm, str(q_loc)), prod_loc)]:
        if key in lu_dict:
            for iid, cnt in lu_dict[key].most_common():
                row = benchmark_items.index.get_loc(benchmark_items.index[benchmark_items["item_id"]==iid][0]) if (benchmark_items["item_id"]==iid).any() else -1
                if row >= 0:
                    pool_set.add(row)

    # Быстрый lookup через item_id -> row index
    if q_norm in prod_gl:
        for iid in prod_gl[q_norm]:
            if iid in VALID_IDS:
                row_arr = np.where(benchmark_items["item_id"].values == iid)[0]
                if len(row_arr): pool_set.add(int(row_arr[0]))

    pool_rows = sorted(pool_set)

    if not return_scores:
        return [ALL_IDS[r] for r in pool_rows], None

    # RRF по всем источникам
    rrf_scores_dict = rrf_score(rank_dicts)

    score_arrays = {
        "bm25": bm25_s, "bm25_geo": bm25_geo_s, "char": char_s,
        "char_geo": char_geo_s, "dense": dense_s, "dense_geo": dense_geo_s,
        "filter_bm25": filter_s,
        "geo_affinity": aff, "geo_reward": geo_rew,
        "same_location": same, "loc_prob": prob, "distance_km": dist,
    }
    return pool_rows, score_arrays, rrf_scores_dict, q_text


# Предвычисляем item_id -> row index для быстрого lookup
_id_to_row = {iid: i for i, iid in enumerate(ALL_IDS)}

def build_pool_fast(q_row):
    """Быстрый вариант pool builder с прямым index lookup."""
    q_text = str(q_row.get("search_query",""))
    q_norm = normalize(q_text)
    q_cat  = str(q_row.get("search_category","") or "")
    q_loc  = _get_loc(q_row)
    q_infm = str(q_row.get("search_infm_params_text","") or "")

    same, prob, dist, aff, geo_rew = compute_geo(q_loc)
    bm25_s = bm25_query_scores(q_text)
    char_s = char_query_scores(q_text)

    sources = {
        "bm25":      bm25_s,
        "bm25_geo":  bm25_s * (1 + 1.25 * aff),
        "char":      char_s,
        "char_geo":  char_s * (1 + aff),
        "filter_bm25": bm25_query_scores(q_infm) if q_infm.strip() else np.zeros(N_ITEMS),
    }
    dense_s = np.zeros(N_ITEMS, dtype=np.float32)
    if dense_model is not None and item_emb is not None:
        q_emb_v = dense_model.encode([f"query: {q_text}"], normalize_embeddings=True,
                                       show_progress_bar=False, convert_to_numpy=True,
                                       device=DEVICE)[0].astype("float32")
        dense_s = query_dense_scores(q_emb_v)
        sources["dense"]     = dense_s
        sources["dense_geo"] = dense_s + geo_rew

    pool_rows = set()
    rr = []
    for src, sarr in sources.items():
        k = TOP.get(src, 100)
        if sarr.max() <= 0: continue
        idxs = topk_indices(sarr, k)
        rr.append({int(idx): rank+1 for rank, idx in enumerate(idxs)})
        pool_rows.update(int(i) for i in idxs)

    # Lookup
    for key, lu in [(q_norm, prod_gl), ((q_norm, q_cat), prod_cat),
                     ((q_norm, str(q_loc)), prod_loc)]:
        for iid in lu.get(key, {}):
            row = _id_to_row.get(iid, -1)
            if row >= 0: pool_rows.add(row)

    rrf_s = rrf_score(rr)

    pool_rows = sorted(pool_rows)
    feat = {
        "bm25_score":      bm25_s[pool_rows],
        "bm25_geo_score":  sources.get("bm25_geo", bm25_s)[pool_rows],
        "char_score":      char_s[pool_rows],
        "char_geo_score":  sources.get("char_geo", char_s)[pool_rows],
        "dense_score":     dense_s[pool_rows],
        "dense_geo_score": sources.get("dense_geo", dense_s)[pool_rows],
        "filter_score":    sources.get("filter_bm25", np.zeros(N_ITEMS))[pool_rows],
        "rrf":             np.array([rrf_s.get(r, 0.0) for r in pool_rows]),
        "geo_affinity":    aff[pool_rows],
        "geo_reward":      geo_rew[pool_rows],
        "same_location":   same[pool_rows],
        "loc_prob":        prob[pool_rows],
        "distance_km":     np.nan_to_num(dist[pool_rows], nan=500.0),
        "log_distance_km": np.log1p(np.nan_to_num(dist[pool_rows], nan=500.0)),
    }
    return pool_rows, feat, q_text, q_norm


def item_features(pool_rows, q_tokens):
    """Признаки объявлений: покрытие, популярность, рейтинг."""
    q_set = set(q_tokens)
    rows_arr  = np.array(pool_rows)
    title_cov = []; desc_cov = []; pop_arr = []; known_arr = []
    rating = []; has_rating = []; log_reviews = []
    for r in pool_rows:
        iid = ALL_IDS[r]
        tt  = set(stem_tokenize(benchmark_items.at[r,"item_title_raw"]))
        dt  = set(stem_tokenize(benchmark_items.at[r,"item_description_raw"][:500]))
        n   = max(len(q_set), 1)
        title_cov.append(len(q_set & tt) / n)
        desc_cov.append(len(q_set & dt) / n)
        pop_arr.append(float(item_pop.get(iid, 0)))
        known_arr.append(float(iid in item_known))
        rat  = benchmark_items.at[r,"item_rating"] if "item_rating" in benchmark_items.columns else np.nan
        rev  = benchmark_items.at[r,"item_rating_reviews_count"] if "item_rating_reviews_count" in benchmark_items.columns else np.nan
        has_rating.append(0.0 if pd.isna(rat) else 1.0)
        rating.append(0.0 if pd.isna(rat) else float(rat))
        log_reviews.append(0.0 if pd.isna(rev) else float(np.log1p(float(rev))))
    return {
        "title_coverage": np.array(title_cov, dtype=np.float32),
        "desc_coverage":  np.array(desc_cov,  dtype=np.float32),
        "popularity":     np.array(pop_arr,    dtype=np.float32),
        "item_known":     np.array(known_arr,  dtype=np.float32),
        "has_rating":     np.array(has_rating, dtype=np.float32),
        "item_rating":    np.array(rating,     dtype=np.float32),
        "log_reviews":    np.array(log_reviews,dtype=np.float32),
    }


def make_feature_matrix(pool_rows, geo_feat, q_tokens):
    """Объединить все признаки в матрицу (n_pool × n_features)."""
    n  = len(pool_rows)
    kw = {**geo_feat, **item_features(pool_rows, q_tokens)}
    cols = sorted(kw.keys())
    mat  = np.column_stack([kw[c] for c in cols]).astype(np.float32)
    # Нормированные ранги внутри запроса
    def rrank(arr):
        order = np.argsort(-arr)
        r = np.empty(n, dtype=np.float32)
        r[order] = 1.0 / (60 + np.arange(1, n+1))
        return r
    rank_cols = {f"rrank_{c}": rrank(kw[c]) for c in ["bm25_score","bm25_geo_score",
                 "dense_score","dense_geo_score","char_score","rrf","geo_affinity"]}
    mat2 = np.column_stack(list(rank_cols.values())).astype(np.float32)
    feat_names = cols + list(rank_cols.keys())
    return np.concatenate([mat, mat2], axis=1), feat_names

## 12. OOD Validation (без реранкера)

In [13]:
print("\nOffline val (пул, без реранкера)...")
q_meta = {}
for _, r in core.iterrows():
    key = normalize(str(r.get("search_query","")))
    if key not in q_meta:
        q_meta[key] = {"loc": r.get("search_location_id",-1), "cat": r.get("search_category","")}

recalls_pre = []
for _, row in tqdm(val_ood.iterrows(), total=len(val_ood), desc="  val-pre", leave=False):
    rel = set(row["relevant_ids"]) & VALID_IDS
    if not rel: continue
    pool_rows, *_ = build_pool_fast(pd.Series({
        "search_query": row["search_query"],
        "search_category": q_meta.get(normalize(row["search_query"]),{}).get("cat",""),
        "search_location_id": q_meta.get(normalize(row["search_query"]),{}).get("loc",-1),
        "search_infm_params_text": "",
    }))
    cands = {ALL_IDS[r] for r in pool_rows[:N_OUT]}
    recalls_pre.append(len(rel & cands) / len(rel))

r_pre = float(np.mean(recalls_pre)) if recalls_pre else 0.0
print(f"  Recall@{N_OUT} [pre-rerank, OOD]: {r_pre:.4f}")
print(f"  Ср. кандидатов: pooling с несколькими источниками -> более полный пул")


Offline val (пул, без реранкера)...


  val-pre:   0%|          | 0/67 [00:00<?, ?it/s]

  Recall@50 [pre-rerank, OOD]: 0.1385
  Ср. кандидатов: pooling с несколькими источниками -> более полный пул


## 13. CatBoost: сборка обучающих данных

In [14]:
X_parts = []; y_parts = []; qid_parts = []
FEAT_NAMES = None
q_to_pos = {}
for _, r in core[core["item_id"].isin(VALID_IDS)].iterrows():
    q_to_pos.setdefault(normalize(str(r["search_query"])), set()).add(r["item_id"])

if HAS_CB:
    print("\nСобираем обучающий пул для CatBoost...")
    train_queries_df = core.drop_duplicates("qnorm").sample(
        frac=1, random_state=SEED
    ).head(8000)

    for qi, (_, tr_row) in enumerate(tqdm(train_queries_df.iterrows(),
                                           total=len(train_queries_df),
                                           desc="  train pool")):
        q_norm = tr_row["qnorm"]
        pos_ids = q_to_pos.get(q_norm, set())
        if not pos_ids: continue

        pool_rows, feat_d, q_text, q_nrm = build_pool_fast(tr_row)
        if not pool_rows: continue

        qtoks = stem_tokenize(q_text)
        feat_mat, fn = make_feature_matrix(pool_rows, feat_d, qtoks)
        if FEAT_NAMES is None: FEAT_NAMES = fn

        labels = np.array([1 if ALL_IDS[r] in pos_ids else 0 for r in pool_rows], dtype=np.int32)
        if labels.sum() == 0: continue

        # Отбор негативов: top-HARD_NEG по RRF + RAND_NEG случайных
        rrf_arr = feat_d["rrf"]
        neg_mask = (labels == 0)
        neg_idx  = np.where(neg_mask)[0]
        if len(neg_idx) > 0:
            hard_idx  = neg_idx[np.argsort(-rrf_arr[neg_idx])[:HARD_NEG]]
            tail_idx  = neg_idx[np.argsort(-rrf_arr[neg_idx])[HARD_NEG:]]
            rand_idx  = (tail_idx[np.random.choice(len(tail_idx),
                         size=min(RAND_NEG, len(tail_idx)), replace=False)]
                         if len(tail_idx) > 0 else np.empty(0, dtype=int))
            keep_mask = np.zeros(len(labels), dtype=bool)
            keep_mask[labels == 1] = True
            keep_mask[hard_idx]    = True
            keep_mask[rand_idx]    = True
            feat_mat = feat_mat[keep_mask]
            labels   = labels[keep_mask]

        X_parts.append(feat_mat)
        y_parts.append(labels)
        qid_parts.append(np.full(len(labels), qi, dtype=np.int32))

if HAS_CB and X_parts:
    X_all  = np.vstack(X_parts)
    y_all  = np.concatenate(y_parts)
    qid_all = np.concatenate(qid_parts)
    pos_rate = float(y_all.mean())
    print(f"  Samples: {len(X_all):,}, positives: {y_all.sum():,} ({pos_rate*100:.2f}%)")


Собираем обучающий пул для CatBoost...


  train pool:   0%|          | 0/8000 [00:00<?, ?it/s]

  Samples: 345,623, positives: 2,855 (0.83%)


## 14. CatBoostRanker: обучение (PairLogit, 2 прохода)

In [15]:
cb_model = None

if HAS_CB and X_parts:
    print("\nCatBoostRanker (PairLogit)...")
    cb1 = CatBoostRanker(
        loss_function="PairLogit:max_pairs=128",
        random_seed=SEED,
        task_type="GPU" if torch.cuda.is_available() else "CPU",
        allow_writing_files=False,
        **RANKER,
    )
    pool1 = CatPool(X_all, label=y_all, group_id=qid_all, feature_names=FEAT_NAMES)
    del X_all, y_all, qid_all; gc.collect()
    cb1.fit(pool1, verbose=50)
    del pool1; gc.collect()

    # Второй проход: hard negatives по прогнозу первой модели
    print("\nВторой проход (hard negatives)...")
    X2_parts=[]; y2_parts=[]; q2_parts=[]
    for qi, (_, tr_row) in enumerate(tqdm(train_queries_df.iterrows(),
                                           total=len(train_queries_df),
                                           desc="  hard neg", leave=True)):
        q_norm = tr_row["qnorm"]
        pos_ids = q_to_pos.get(q_norm, set())
        if not pos_ids: continue
        pool_rows, feat_d, q_text, q_nrm = build_pool_fast(tr_row)
        if not pool_rows: continue
        qtoks   = stem_tokenize(q_text)
        feat_mat, _ = make_feature_matrix(pool_rows, feat_d, qtoks)
        labels  = np.array([1 if ALL_IDS[r] in pos_ids else 0 for r in pool_rows], dtype=np.int32)
        if labels.sum() == 0: continue
        # Hard negatives: top-HARD_NEG по прогнозу первой модели
        scores_1 = cb1.predict(feat_mat, thread_count=-1)
        neg_mask = (labels == 0)
        neg_idx  = np.where(neg_mask)[0]
        if len(neg_idx) > 0:
            hard_idx = neg_idx[np.argsort(-scores_1[neg_idx])[:HARD_NEG]]
            tail_idx = neg_idx[np.argsort(-scores_1[neg_idx])[HARD_NEG:]]
            rand_idx = (tail_idx[np.random.choice(len(tail_idx),
                        size=min(RAND_NEG, len(tail_idx)), replace=False)]
                        if len(tail_idx) > 0 else np.empty(0, dtype=int))
            keep_mask = np.zeros(len(labels), dtype=bool)
            keep_mask[labels==1]=True; keep_mask[hard_idx]=True; keep_mask[rand_idx]=True
            feat_mat = feat_mat[keep_mask]; labels = labels[keep_mask]
        X2_parts.append(feat_mat); y2_parts.append(labels)
        q2_parts.append(np.full(len(labels), qi, dtype=np.int32))

    X2   = np.vstack(X2_parts)
    y2   = np.concatenate(y2_parts)
    qid2 = np.concatenate(q2_parts)
    print(f"  Второй проход: {len(X2):,} samples")
    cb_model = CatBoostRanker(
        loss_function="PairLogit:max_pairs=128",
        random_seed=SEED,
        task_type="GPU" if torch.cuda.is_available() else "CPU",
        allow_writing_files=False,
        **RANKER,
    )
    pool2 = CatPool(X2, label=y2, group_id=qid2, feature_names=FEAT_NAMES)
    del X2, y2, qid2; gc.collect()
    cb_model.fit(pool2, verbose=50)
    del pool2; gc.collect()
    cb_model.save_model(str(SAVE_DIR/"ranker.cbm"))
    print("  -> ranker.cbm")
    # fi = dict(zip(FEAT_NAMES, cb_model.get_feature_importance()))
    # print("\n  Топ признаков:")
    # for fn, fv in sorted(fi.items(), key=lambda x:-x[1])[:12]:
    #     print(f"    {fn:<25} {fv:.2f}")
else:
    print("[INFO] CatBoost недоступен или нет данных")


CatBoostRanker (PairLogit)...
0:	learn: 0.4399046	total: 217ms	remaining: 1m 5s
50:	learn: 0.0339459	total: 702ms	remaining: 3.43s
100:	learn: 0.0293627	total: 1.09s	remaining: 2.16s
150:	learn: 0.0258219	total: 1.5s	remaining: 1.48s
200:	learn: 0.0232296	total: 1.88s	remaining: 928ms
250:	learn: 0.0209147	total: 2.29s	remaining: 448ms
299:	learn: 0.0190558	total: 2.67s	remaining: 0us

Второй проход (hard negatives)...


  hard neg:   0%|          | 0/8000 [00:00<?, ?it/s]

  Второй проход: 345,623 samples
0:	learn: 0.4620929	total: 13.9ms	remaining: 4.15s
50:	learn: 0.0472774	total: 505ms	remaining: 2.47s
100:	learn: 0.0391386	total: 920ms	remaining: 1.81s
150:	learn: 0.0348127	total: 1.47s	remaining: 1.45s
200:	learn: 0.0306814	total: 2.54s	remaining: 1.25s
250:	learn: 0.0276859	total: 3.73s	remaining: 728ms
299:	learn: 0.0253931	total: 4.24s	remaining: 0us
  -> ranker.cbm


CatBoostError: Feature importance type EFstrType.LossFunctionChange requires training dataset                             to be passed to this function.

## 15. Post-rerank validation

In [16]:
r_post = r_pre

if cb_model is not None:
    print("\nOffline val (с реранкером)...")
    recalls_post = []
    for _, row in tqdm(val_ood.iterrows(), total=len(val_ood),
                        desc="  val-post", leave=False):
        rel = set(row["relevant_ids"]) & VALID_IDS
        if not rel: continue
        meta = q_meta.get(normalize(row["search_query"]), {})
        qr = pd.Series({"search_query": row["search_query"],
                         "search_category": meta.get("cat",""),
                         "search_location_id": meta.get("loc",-1),
                         "search_infm_params_text": ""})
        pool_rows, feat_d, q_text, _ = build_pool_fast(qr)
        if not pool_rows: recalls_post.append(0.0); continue
        feat_mat, _ = make_feature_matrix(pool_rows, feat_d, stem_tokenize(q_text))
        probs = cb_model.predict(feat_mat, thread_count=-1)
        top50 = [ALL_IDS[pool_rows[j]] for j in np.argsort(-probs)[:N_OUT]]
        recalls_post.append(len(rel & set(top50)) / len(rel))

    r_post = float(np.mean(recalls_post)) if recalls_post else 0.0
    print(f"\n  Recall@{N_OUT} [pre-rerank]:  {r_pre:.4f}")
    print(f"  Recall@{N_OUT} [post-rerank]: {r_post:.4f}  ({(r_post-r_pre)*100:+.2f} п.п.)")


Offline val (с реранкером)...


  val-post:   0%|          | 0/67 [00:00<?, ?it/s]


  Recall@50 [pre-rerank]:  0.1385
  Recall@50 [post-rerank]: 0.5665  (+42.80 п.п.)


## 16. Генерация answer.csv

In [17]:
print(f"\nГенерируем {len(benchmark_queries):,} запросов...")

# Batch encode benchmark queries
bench_q_embs = None
if dense_model is not None and faiss_idx is not None:
    print("  Batch encode queries...")
    bench_texts = [f"query: {r.get('search_query','')}" for _,r in benchmark_queries.iterrows()]
    bench_q_embs = dense_model.encode(
        bench_texts, batch_size=ENC_BATCH, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    ).astype("float32")

all_qids=[]; all_preds=[]
for i, (_, q_row) in enumerate(tqdm(benchmark_queries.iterrows(),
                                     total=len(benchmark_queries), desc="queries")):
    qid = str(q_row["query_id"])
    pool_rows, feat_d, q_text, _ = build_pool_fast(q_row)

    if cb_model is not None and pool_rows:
        feat_mat, _ = make_feature_matrix(pool_rows, feat_d, stem_tokenize(q_text))
        probs = cb_model.predict(feat_mat, thread_count=-1)
        top50 = [ALL_IDS[pool_rows[j]] for j in np.argsort(-probs)[:N_OUT]
                 if ALL_IDS[pool_rows[j]] in VALID_IDS][:N_OUT]
    else:
        # Fallback: RRF-ранжирование без реранкера
        rrf_a = feat_d.get("rrf", np.zeros(len(pool_rows)))
        top50 = [ALL_IDS[pool_rows[j]] for j in np.argsort(-rrf_a)[:N_OUT]
                 if ALL_IDS[pool_rows[j]] in VALID_IDS][:N_OUT]

    all_qids.append(qid); all_preds.append(top50)


Генерируем 2,452 запросов...
  Batch encode queries...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

queries:   0%|          | 0/2452 [00:00<?, ?it/s]